<a href="https://colab.research.google.com/github/harrisonvills/analisis-perovskita-python/blob/main/analisis_estadistico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# 1. Cargar el archivo
# Asegúrate de que el archivo 'Descriptors_Perovskite_Database.xlsx'
# esté subido en la pestaña de archivos a la izquierda.
file_path = 'Descriptors_Perovskite_Database.xlsx'
df = pd.read_excel(file_path)

# 2. Definir el vector objetivo completo para el material MAPbI3
target_vector = [-0.0360071706690867, 0.00658662108566845, 0.0175155451920533,
                 -0.00138823269521618]

# 3. Filtrar los datos incluyendo las 4 columnas (L1, L2, L3, L4)
# Usamos np.isclose para asegurar una comparación precisa entre flotantes
mask = np.isclose(df[['LLE_1', 'LLE_2', 'LLE_3', 'LLE_4']], target_vector, atol=1e-5).all(axis=1)
df_mapbi3 = df[mask].copy()

# 4. Seleccionar las variables requeridas para el análisis
cols_to_use = [
    'DMF_DMSO_ratio',
    'Perovskite_annealing_thermal_exposure',
    'first_Prvskt_annealing_temperature',
    'Perovskite_thickness',
    'JV_default_PCE'
]

# Crear el DataFrame limpio
df_final = df_mapbi3[cols_to_use]

# Verificar resultado
print("Dimensiones del dataset filtrado (filas, columnas):", df_final.shape)
print(df_final.head())

Dimensiones del dataset filtrado (filas, columnas): (26320, 5)
    DMF_DMSO_ratio  Perovskite_annealing_thermal_exposure  \
9         4.000000                               3.819544   
10        4.000000                               3.819544   
11        4.000000                               3.819544   
15        0.953808                               2.939519   
16        0.953808                               2.939519   

    first_Prvskt_annealing_temperature Perovskite_thickness  JV_default_PCE  
9                                110.0                  NaN            10.3  
10                               110.0                  NaN            11.5  
11                               110.0                  NaN            12.9  
15                                70.0                400.0            12.0  
16                                70.0                400.0             9.0  


In [ ]:
# 1. Eliminar filas que tengan datos vacíos (NaN) en cualquiera de las columnas
df_sin_nan = df_final.dropna()

# 2. Calcular el Rango Intercuartílico (IQR) para la eficiencia (JV_default_PCE)
Q1 = df_sin_nan['JV_default_PCE'].quantile(0.25)
Q3 = df_sin_nan['JV_default_PCE'].quantile(0.75)
IQR = Q3 - Q1

# 3. Definir los límites inferior y superior para detectar anomalías
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 4. Filtrar el DataFrame final
# Nos quedamos con valores dentro del límite y que sean mayores o iguales a 0
df_clean = df_sin_nan[
    (df_sin_nan['JV_default_PCE'] >= 0) &
    (df_sin_nan['JV_default_PCE'] >= lower_bound) &
    (df_sin_nan['JV_default_PCE'] <= upper_bound)
].copy()

# Verificar cuántos datos perfectos nos quedan para trabajar
print(f"Registros antes de limpieza profunda: {len(df_final)}")
print(f"Registros listos para el análisis estadístico: {len(df_clean)}")

Registros antes de limpieza profunda: 26320
Registros listos para el análisis estadístico: 6131


FASE II

In [ ]:
import scipy.stats as stats
import pandas as pd

print("--- 0. CORRECCIÓN DE TIPO DE DATOS ---")
# Lista de nuestras columnas
columnas_numericas = [
    'DMF_DMSO_ratio',
    'Perovskite_annealing_thermal_exposure',
    'first_Prvskt_annealing_temperature',
    'Perovskite_thickness',
    'JV_default_PCE'
]

# Forzamos a que todas sean tratadas como números (float)
for col in columnas_numericas:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Si algún texto no se pudo convertir, se vuelve NaN. Los limpiamos por seguridad:
df_clean = df_clean.dropna()
print("Datos convertidos exitosamente a formato numérico.")


print("\n--- 1. CORRELACIÓN DE PEARSON Y PRUEBA DE HIPÓTESIS ---")
# Calculamos la matriz de correlación de Pearson
matriz_corr = df_clean.corr(method='pearson')
print("Correlación con la Eficiencia (JV_default_PCE):\n")
print(matriz_corr['JV_default_PCE'].drop('JV_default_PCE', errors='ignore'))

# Prueba de hipótesis para cada variable vs la eficiencia [cite: 52, 53]
print("\nResultados de la prueba de hipótesis (p-value):")
columnas_entrada = ['DMF_DMSO_ratio', 'Perovskite_annealing_thermal_exposure',
                    'first_Prvskt_annealing_temperature', 'Perovskite_thickness']

for col in columnas_entrada:
    corr, p_value = stats.pearsonr(df_clean[col], df_clean['JV_default_PCE'])
    print(f"\nVariable: {col}")
    print(f"Coeficiente r: {corr:.4f} | p-value: {p_value:.4e}")
    if p_value < 0.05:
         print("-> Correlación estadísticamente significativa.")
    else:
         print("-> No hay correlación significativa.")

print("\n---------------------------------------------------")
print("--- 2. PRUEBA DE INDEPENDENCIA (CHI-CUADRADO) ---")

# Discretizar temperatura usando la mediana ("Alta" vs "Baja") [cite: 54]
mediana_temp = df_clean['first_Prvskt_annealing_temperature'].median()
df_clean.loc[:, 'Temp_Cat'] = ['Alta' if x >= mediana_temp else 'Baja' for x in df_clean['first_Prvskt_annealing_temperature']]

# Discretizar espesor usando la mediana ("Gruesa" vs "Delgada") [cite: 54]
mediana_espesor = df_clean['Perovskite_thickness'].median()
df_clean.loc[:, 'Espesor_Cat'] = ['Gruesa' if x >= mediana_espesor else 'Delgada' for x in df_clean['Perovskite_thickness']]

# Crear tabla de contingencia
tabla_contingencia = pd.crosstab(df_clean['Temp_Cat'], df_clean['Espesor_Cat'])
print("\nTabla de Contingencia (Frecuencias):\n", tabla_contingencia)

# Aplicar prueba de Chi-cuadrado [cite: 54]
chi2, p_val_chi2, dof, expected = stats.chi2_contingency(tabla_contingencia)
print(f"\nEstadístico Chi-cuadrado: {chi2:.4f} | p-value: {p_val_chi2:.4e}")

if p_val_chi2 < 0.05:
    print("-> Conclusión: Rechazamos H0. Existe dependencia estadística entre el espesor y la temperatura de recocido.")
else:
    print("-> Conclusión: No rechazamos H0. Son estadísticamente independientes.")

--- 0. CORRECCIÓN DE TIPO DE DATOS ---
Datos convertidos exitosamente a formato numérico.

--- 1. CORRELACIÓN DE PEARSON Y PRUEBA DE HIPÓTESIS ---
Correlación con la Eficiencia (JV_default_PCE):

DMF_DMSO_ratio                          -0.231101
Perovskite_annealing_thermal_exposure   -0.185063
first_Prvskt_annealing_temperature       0.050141
Perovskite_thickness                     0.017203
Name: JV_default_PCE, dtype: float64

Resultados de la prueba de hipótesis (p-value):

Variable: DMF_DMSO_ratio
Coeficiente r: -0.2311 | p-value: 3.9321e-75
-> Correlación estadísticamente significativa.

Variable: Perovskite_annealing_thermal_exposure
Coeficiente r: -0.1851 | p-value: 2.2878e-48
-> Correlación estadísticamente significativa.

Variable: first_Prvskt_annealing_temperature
Coeficiente r: 0.0501 | p-value: 8.5738e-05
-> Correlación estadísticamente significativa.

Variable: Perovskite_thickness
Coeficiente r: 0.0172 | p-value: 1.7803e-01
-> No hay correlación significativa.

--------

FASE III

In [ ]:
import scipy.stats as stats
import numpy as np

print("--- 3. PRUEBA DE HIPÓTESIS: PROPORCIÓN DE SOLVENTES (DMF/DMSO) ---")

# 1. Definir los umbrales para los grupos usando percentiles (25% y 75%)
q25 = df_clean['DMF_DMSO_ratio'].quantile(0.25)
q75 = df_clean['DMF_DMSO_ratio'].quantile(0.75)

print(f"Límites definidos -> Valores Bajos: <= {q25:.2f} | Valores Altos: >= {q75:.2f}")

# 2. Dividir las observaciones en los dos grupos solicitados
# Grupo 1: Valores bajos ó altos (extremos)
grupo1 = df_clean[(df_clean['DMF_DMSO_ratio'] <= q25) | (df_clean['DMF_DMSO_ratio'] >= q75)]['JV_default_PCE']

# Grupo 2: Valores promedio (intermedios)
grupo2 = df_clean[(df_clean['DMF_DMSO_ratio'] > q25) & (df_clean['DMF_DMSO_ratio'] < q75)]['JV_default_PCE']

print(f"Tamaño Grupo 1 (Extremos): {len(grupo1)} observaciones")
print(f"Tamaño Grupo 2 (Promedio): {len(grupo2)} observaciones")

# 3. Ejecutar la prueba T de Student para muestras independientes
# Usamos equal_var=False (Prueba de Welch) porque los grupos tienen tamaños distintos
t_stat, p_val_t = stats.ttest_ind(grupo1, grupo2, equal_var=False)

print(f"\nResultados de la prueba T de Student:")
print(f"Estadístico t: {t_stat:.4f} | p-value: {p_val_t:.4e}")

# 4. Decisión formal
alpha = 0.05
print("\nPlanteamiento Formal:")
print("H0 = μ1 = μ2 (La eficiencia media es igual en ambos grupos)")
print("H1 = μ1 ≠ μ2 (La eficiencia media cambia significativamente)")

if p_val_t < alpha:
    print("\n-> Decisión: Se RECHAZA la hipótesis nula (H0).")
    print("-> Conclusión: La proporción de solventes SÍ influye en el rendimiento de la celda solar.")
else:
    print("\n-> Decisión: NO se rechaza la hipótesis nula (H0).")
    print("-> Conclusión: No hay evidencia de que la proporción de solventes influya significativamente.")

--- 3. PRUEBA DE HIPÓTESIS: PROPORCIÓN DE SOLVENTES (DMF/DMSO) ---
Límites definidos -> Valores Bajos: <= 0.37 | Valores Altos: >= 4.00
Tamaño Grupo 1 (Extremos): 3594 observaciones
Tamaño Grupo 2 (Promedio): 2537 observaciones

Resultados de la prueba T de Student:
Estadístico t: -10.0446 | p-value: 1.5776e-23

Planteamiento Formal:
H0 = μ1 = μ2 (La eficiencia media es igual en ambos grupos)
H1 = μ1 ≠ μ2 (La eficiencia media cambia significativamente)

-> Decisión: Se RECHAZA la hipótesis nula (H0).
-> Conclusión: La proporción de solventes SÍ influye en el rendimiento de la celda solar.


FASE IV


In [ ]:
import statsmodels.api as sm

print("--- 4. MODELO DE REGRESIÓN LINEAL MÚLTIPLE ---")

# 1. Definir las variables de entrada (X) y la de salida (Y)
X = df_clean[[
    'DMF_DMSO_ratio',
    'Perovskite_annealing_thermal_exposure',
    'first_Prvskt_annealing_temperature',
    'Perovskite_thickness'
]]
Y = df_clean['JV_default_PCE']

# 2. Añadir la constante (el beta_0 de la ecuación)
X = sm.add_constant(X)

# 3. Ajustar el modelo matemático (Ordinary Least Squares - OLS)
modelo = sm.OLS(Y, X).fit()

# 4. Imprimir el súper resumen con todos los datos que pide el archivo
print(modelo.summary())

--- 4. MODELO DE REGRESIÓN LINEAL MÚLTIPLE ---
                            OLS Regression Results                            
Dep. Variable:         JV_default_PCE   R-squared:                       0.069
Model:                            OLS   Adj. R-squared:                  0.069
Method:                 Least Squares   F-statistic:                     113.9
Date:                Sun, 24 May 2026   Prob (F-statistic):           8.23e-94
Time:                        23:50:30   Log-Likelihood:                -17645.
No. Observations:                6131   AIC:                         3.530e+04
Df Residuals:                    6126   BIC:                         3.533e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                                            coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------